# Card Builder
Runs the card builder independently.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name == 'certification_framework' else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import json
from scripts.computation.card_builder import build_from_file

NB_DATA = ROOT / 'notebooks' / 'data'
phase1_path = NB_DATA / 'parsed_context.json'
result = build_from_file(phase1_path)

# Save to notebook-local data
(NB_DATA / 'cards.json').write_text(
    json.dumps(result, indent=2, default=str, ensure_ascii=False), encoding='utf-8'
)

print(f'=== Cards ({len(result["cards"])}) ===')
for name, card in result['cards'].items():
    print(f'\n  {name}: {card.get("title", "untitled")}')
    for item in card['items']:
        print(f'    {item.get("label", "?")}: {item.get("value", "?")}')

In [ ]:
raw = json.loads((ROOT / 'data' / 'input' / 'aggregated_scorecard_output.json').read_text(encoding='utf-8'))

print('=== Validation: Cards vs Input ===')
all_ok = True

# Identity card should have agent name
identity = result['cards'].get('identity_card', {})
agent_name = None
for item in identity.get('items', []):
    if item.get('label') == 'Agent Name':
        agent_name = item['value']
ok = 'PASS' if agent_name == raw['agent_name'] else 'FAIL'
if ok == 'FAIL': all_ok = False
print(f'  {ok} identity_card agent_name: {agent_name} (expected {raw["agent_name"]})')

# Expected card names
expected_cards = ['identity_card', 'scope_card', 'categories_card']
for name in expected_cards:
    ok = 'PASS' if name in result['cards'] else 'FAIL'
    if ok == 'FAIL': all_ok = False
    print(f'  {ok} card present: {name}')

print(f'\nResult: {"ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED"}')